# Logistic Regression Classifier

7-class classifier (multinomial logistic regression) — the smallest possible model here: just one weight vector per class.

In [ ]:
# ==========================================================
# CELL 0 — install dependencies
# ==========================================================
%pip install pandas numpy scikit-learn joblib openpyxl matplotlib skl2onnx onnx


In [ ]:
# ==========================================================
# CELL 1 — imports
# ==========================================================
import os
import glob
import random
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


In [ ]:
# ==========================================================
# CELL 2 — configuration
# ==========================================================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

BASE_PATH = (
    r"../../data"
    r"\dataset\Motor_data_model"
)

SCALE_DIVISOR = 1_000_000.0

FEATURE_COLUMNS = [
    "T_slope_10s", "PWM", "Irms", "Vrms",
    "Current_Variance", "Current_Crest_Factor",
    "Vib_RMS", "Vib_Min", "Vib_Q1", "Vib_Median",
    "Vib_Q3", "Vib_Max", "Vib_Variance", "Vib_Skewness",
]

FINAL_LABELS = [
    "motor_off", "turning_off", "turning_on", "normal",
    "electrical_fault", "mechanical_fault", "thermal_fault",
]

LABEL_TO_ID = {
    label: index
    for index, label in enumerate(FINAL_LABELS)
}

N_FEATURES = len(FEATURE_COLUMNS)
N_CLASSES = len(FINAL_LABELS)

WINDOW_LEN = 30
WINDOW_STEP = 10
INPUT_SIZE = WINDOW_LEN * N_FEATURES
ROW_PERIOD_SEC = 0.015
N_SPLITS = 5

OUTPUT_DIR = "logistic_regression_output"
CV_DIR = os.path.join(OUTPUT_DIR, "cross_validation")
FINAL_DIR = os.path.join(OUTPUT_DIR, "final_model")
MANUAL_TEST_DIR = os.path.join(OUTPUT_DIR, "manual_test_results")

for folder in [CV_DIR, FINAL_DIR, MANUAL_TEST_DIR]:
    os.makedirs(folder, exist_ok=True)

print("Model: Logistic Regression Classifier")
print("Input:", WINDOW_LEN, "x", N_FEATURES)
print("Classes:", FINAL_LABELS)


In [ ]:
# ==========================================================
# CELL 3 — map files to classes
# ==========================================================
def normalized_path(path):
    return str(path).replace("\\", "/").lower()


def label_for(path):
    path_text = normalized_path(path)
    stem = Path(path).stem.lower()

    if any(term in path_text for term in [
        "motor_being_off",
        "turning_off",
        "transition_off",
        "off_spikes",
    ]):
        return "turning_off"

    if any(term in path_text for term in [
        "motor_being_on",
        "turning_on",
        "transition_on",
        "on_spikes",
    ]):
        return "turning_on"

    if "/motor_off/" in path_text or "motor_off" in stem:
        return "motor_off"

    if any(term in path_text for term in [
        "electrical fault",
        "electrical_fault",
        "abnormal_current",
    ]):
        return "electrical_fault"

    if any(term in path_text for term in [
        "mechanical fault",
        "mechanical_fault",
        "abnormal_vibration",
    ]):
        return "mechanical_fault"

    if any(term in path_text for term in [
        "thermal fault",
        "thermal_fault",
        "abnormal_temp",
    ]):
        return "thermal_fault"

    if (
        "/normal/" in path_text
        or stem.startswith("normal_")
        or stem == "normal"
    ):
        return "normal"

    return None



In [ ]:
# ==========================================================
# CELL 4 — discover and read files
# ==========================================================
def discover_files(base_path):
    files = glob.glob(
        os.path.join(base_path, "**", "*.xlsx"),
        recursive=True,
    )
    files += glob.glob(
        os.path.join(base_path, "**", "*.csv"),
        recursive=True,
    )
    return sorted(set(files))


def read_one(path):
    path = Path(path)

    if path.suffix.lower() == ".xlsx":
        df = pd.read_excel(path)
    else:
        df = pd.read_csv(path)

    missing_columns = [
        column
        for column in FEATURE_COLUMNS
        if column not in df.columns
    ]

    if missing_columns:
        raise ValueError(
            f"Missing columns in {path.name}: {missing_columns}"
        )

    clean = df[FEATURE_COLUMNS].apply(
        pd.to_numeric,
        errors="coerce",
    )

    clean = clean / SCALE_DIVISOR
    clean = clean.replace([np.inf, -np.inf], np.nan)
    clean = clean.dropna().reset_index(drop=True)

    if len(clean) < WINDOW_LEN:
        raise ValueError(
            f"{path.name} has only {len(clean)} valid rows."
        )

    return clean



In [ ]:
# ==========================================================
# CELL 5 — load labeled sessions
# ==========================================================
files = discover_files(BASE_PATH)

if not files:
    raise RuntimeError(
        f"No CSV or Excel files found under: {BASE_PATH}"
    )

sessions = []

for file_path in files:
    label = label_for(file_path)

    if label is None:
        print(
            "SKIP unrecognized:",
            os.path.relpath(file_path, BASE_PATH),
        )
        continue

    try:
        df = read_one(file_path)
    except Exception as error:
        print(
            "SKIP read error:",
            os.path.relpath(file_path, BASE_PATH),
            "|",
            error,
        )
        continue

    sessions.append({
        "X": df.to_numpy(dtype=np.float32),
        "file": os.path.relpath(file_path, BASE_PATH),
        "label": label,
        "class_id": LABEL_TO_ID[label],
        "rows": len(df),
    })

if not sessions:
    raise RuntimeError("No labeled sessions were loaded.")

class_counts = pd.Series([
    session["label"]
    for session in sessions
]).value_counts().reindex(
    FINAL_LABELS,
    fill_value=0,
)

print("\nFILES PER CLASS")
print(class_counts)

missing_classes = [
    label
    for label in FINAL_LABELS
    if class_counts[label] == 0
]

if missing_classes:
    raise RuntimeError(
        f"No files found for classes: {missing_classes}"
    )


In [ ]:
# ==========================================================
# CELL 6 — create flattened 30 × 14 windows
# ==========================================================
def make_windows_from_array(X_raw):
    starts = np.arange(
        0,
        len(X_raw) - WINDOW_LEN + 1,
        WINDOW_STEP,
        dtype=np.int64,
    )

    if len(starts) == 0:
        return None, None

    windows = np.stack([
        X_raw[start:start + WINDOW_LEN]
        for start in starts
    ]).astype(np.float32)

    return (
        windows.reshape(len(windows), INPUT_SIZE),
        starts,
    )


def build_dataset_from_sessions(selected_sessions):
    all_windows = []
    all_labels = []
    all_files = []
    all_starts = []

    for session in selected_sessions:
        windows, starts = make_windows_from_array(
            session["X"]
        )

        if windows is None:
            continue

        all_windows.append(windows)
        all_labels.extend(
            [session["class_id"]] * len(windows)
        )
        all_files.extend(
            [session["file"]] * len(windows)
        )
        all_starts.extend(starts.tolist())

    if not all_windows:
        raise RuntimeError("No windows were created.")

    return (
        np.concatenate(all_windows, axis=0),
        np.asarray(all_labels, dtype=np.int64),
        np.asarray(all_files, dtype=object),
        np.asarray(all_starts, dtype=np.int64),
    )



In [ ]:
# ==========================================================
# CELL 7 — flattened feature names
# ==========================================================
FLAT_FEATURE_NAMES = [
    f"t{time_index:02d}_{feature_name}"
    for time_index in range(WINDOW_LEN)
    for feature_name in FEATURE_COLUMNS
]


In [ ]:
# ==========================================================
# CELL 8 — build Logistic Regression classifier
# ==========================================================
def build_classifier():
    return Pipeline([
        (
            "scaler",
            StandardScaler(),
        ),
        (
            "classifier",
            LogisticRegression(
                penalty="l2",
                C=1.0,
                solver="lbfgs",
                max_iter=5000,
                class_weight="balanced",
                random_state=42,
            ),
        ),
    ])


temporary_model = build_classifier()

print("Logistic Regression configuration:")
print(temporary_model.get_params())

del temporary_model


In [ ]:
# ==========================================================
# CELL 9 — five-fold file-wise cross-validation
# ==========================================================
cv_table = pd.DataFrame({
    "label": [
        session["class_id"]
        for session in sessions
    ],
    "group": [
        session["file"]
        for session in sessions
    ],
})

splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

cv_metrics = []
all_true = []
all_pred = []
all_window_predictions = []

for fold, (train_index, test_index) in enumerate(
    splitter.split(
        cv_table,
        cv_table["label"],
        groups=cv_table["group"],
    ),
    start=1,
):
    print("\n" + "=" * 60)
    print("FOLD", fold)
    print("=" * 60)

    train_sessions = [
        sessions[index]
        for index in train_index
    ]

    test_sessions = [
        sessions[index]
        for index in test_index
    ]

    X_train, y_train, _, _ = (
        build_dataset_from_sessions(train_sessions)
    )

    (
        X_test,
        y_test,
        test_files,
        test_starts,
    ) = build_dataset_from_sessions(test_sessions)

    model = build_classifier()
    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    probabilities = model.predict_proba(X_test)
    confidence = np.max(probabilities, axis=1)

    all_true.extend(y_test.tolist())
    all_pred.extend(predictions.tolist())

    fold_metrics = {
        "fold": fold,
        "accuracy": accuracy_score(
            y_test,
            predictions,
        ),
        "balanced_accuracy": balanced_accuracy_score(
            y_test,
            predictions,
        ),
        "macro_precision": precision_score(
            y_test,
            predictions,
            average="macro",
            zero_division=0,
        ),
        "macro_recall": recall_score(
            y_test,
            predictions,
            average="macro",
            zero_division=0,
        ),
        "macro_f1": f1_score(
            y_test,
            predictions,
            average="macro",
            zero_division=0,
        ),
    }

    cv_metrics.append(fold_metrics)

    all_window_predictions.append(
        pd.DataFrame({
            "fold": fold,
            "file": test_files,
            "start_row": test_starts,
            "true_class_id": y_test,
            "true_label": [
                FINAL_LABELS[index]
                for index in y_test
            ],
            "pred_class_id": predictions,
            "pred_label": [
                FINAL_LABELS[index]
                for index in predictions
            ],
            "confidence": confidence,
        })
    )

    print(pd.Series(fold_metrics))

In [ ]:
# ==========================================================
# CELL 10 — combined cross-validation report
# ==========================================================
cv_metrics_df = pd.DataFrame(cv_metrics)

cv_predictions_df = pd.concat(
    all_window_predictions,
    ignore_index=True,
)

cv_metrics_df.to_csv(
    os.path.join(CV_DIR, "cv_metrics.csv"),
    index=False,
)

cv_predictions_df.to_csv(
    os.path.join(CV_DIR, "cv_window_predictions.csv"),
    index=False,
)

all_true = np.asarray(all_true, dtype=np.int64)
all_pred = np.asarray(all_pred, dtype=np.int64)

overall_metrics = {
    "accuracy": accuracy_score(
        all_true,
        all_pred,
    ),
    "balanced_accuracy": balanced_accuracy_score(
        all_true,
        all_pred,
    ),
    "macro_precision": precision_score(
        all_true,
        all_pred,
        average="macro",
        zero_division=0,
    ),
    "macro_recall": recall_score(
        all_true,
        all_pred,
        average="macro",
        zero_division=0,
    ),
    "macro_f1": f1_score(
        all_true,
        all_pred,
        average="macro",
        zero_division=0,
    ),
}

print("\nOVERALL CROSS-VALIDATION RESULTS")
print(pd.Series(overall_metrics))

print(
    classification_report(
        all_true,
        all_pred,
        labels=np.arange(N_CLASSES),
        target_names=FINAL_LABELS,
        zero_division=0,
    )
)

pd.DataFrame([
    overall_metrics
]).to_csv(
    os.path.join(CV_DIR, "overall_metrics.csv"),
    index=False,
)

ConfusionMatrixDisplay(
    confusion_matrix(
        all_true,
        all_pred,
        labels=np.arange(N_CLASSES),
    ),
    display_labels=FINAL_LABELS,
).plot(
    values_format="d",
    xticks_rotation=45,
)

plt.title(
    "Logistic Regression Classifier Cross-Validation"
)

plt.tight_layout()

plt.savefig(
    os.path.join(CV_DIR, "confusion_matrix.png"),
    dpi=180,
)

plt.show()



In [ ]:
# ==========================================================
# CELL 11 — train final model on all sessions
# ==========================================================
X_all, y_all, _, _ = (
    build_dataset_from_sessions(sessions)
)

final_model = Pipeline([
    (
        "scaler",
        StandardScaler(),
    ),
    (
        "classifier",
        LogisticRegression(
            penalty="l2",
            C=1.0,
            solver="lbfgs",
            max_iter=5000,
            class_weight="balanced",
            random_state=42,
        ),
    ),
])

final_model.fit(X_all, y_all)

print(
    "Final model trained on windows:",
    len(X_all),
)


In [ ]:
# ==========================================================
# CELL 12 — save model, labels, and metadata
# ==========================================================
MODEL_PATH = os.path.abspath(
    os.path.join(
        FINAL_DIR,
        "logistic_regression_classifier.pkl",
    )
)

LABELS_PATH = os.path.abspath(
    os.path.join(
        FINAL_DIR,
        "logistic_regression_labels.txt",
    )
)

METADATA_PATH = os.path.abspath(
    os.path.join(
        FINAL_DIR,
        "logistic_regression_metadata.csv",
    )
)

joblib.dump(final_model, MODEL_PATH)

with open(
    LABELS_PATH,
    "w",
    encoding="utf-8",
) as file:
    file.write("\n".join(FINAL_LABELS))

pd.DataFrame({
    "flat_feature_id": np.arange(INPUT_SIZE),
    "flat_feature_name": FLAT_FEATURE_NAMES,
}).to_csv(
    METADATA_PATH,
    index=False,
)

print("Saved model:", MODEL_PATH)
print(
    "Model size KB:",
    round(os.path.getsize(MODEL_PATH) / 1024, 2),
)
print("Saved labels:", LABELS_PATH)
print("Saved metadata:", METADATA_PATH)

In [ ]:
# ==========================================================
# CELL 13 — coefficient importance
# ==========================================================
classifier = final_model.named_steps[
    "classifier"
]

mean_absolute_coefficient = np.mean(
    np.abs(classifier.coef_),
    axis=0,
)

importance_df = pd.DataFrame({
    "flat_feature_name": FLAT_FEATURE_NAMES,
    "mean_absolute_coefficient":
        mean_absolute_coefficient,
})

importance_df = importance_df.sort_values(
    "mean_absolute_coefficient",
    ascending=False,
).reset_index(drop=True)

importance_df.to_csv(
    os.path.join(
        FINAL_DIR,
        "coefficient_importance.csv",
    ),
    index=False,
)

print("\nTOP 30 FEATURES")
print(importance_df.head(30))


In [ ]:
# ==========================================================
# CELL 14 — export ONNX
# ==========================================================
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

ONNX_PATH = os.path.abspath(
    os.path.join(
        FINAL_DIR,
        "logistic_regression_classifier.onnx",
    )
)

onnx_model = convert_sklearn(
    final_model,
    initial_types=[
        (
            "float_input",
            FloatTensorType([
                None,
                INPUT_SIZE,
            ]),
        )
    ],
    target_opset=13,
)

with open(ONNX_PATH, "wb") as file:
    file.write(
        onnx_model.SerializeToString()
    )

print("Saved:", ONNX_PATH)
print(
    "ONNX size KB:",
    round(os.path.getsize(ONNX_PATH) / 1024, 2),
    
)

In [ ]:
# ==========================================================
# CELL 15 — export direct C header
# ==========================================================
HEADER_PATH = os.path.abspath(
    os.path.join(
        FINAL_DIR,
        "logistic_regression_classifier_model.h",
    )
)

scaler = final_model.named_steps[
    "scaler"
]

classifier = final_model.named_steps[
    "classifier"
]

scaler_mean = scaler.mean_.astype(np.float32)
scaler_scale = scaler.scale_.astype(np.float32)
weights = classifier.coef_.astype(np.float32)
bias = classifier.intercept_.astype(np.float32)


def write_float_array(
    file,
    name,
    values,
):
    values = np.asarray(
        values,
        dtype=np.float32,
    ).reshape(-1)

    file.write(
        f"static const float {name}"
        f"[{len(values)}] = {{\n"
    )

    for start in range(0, len(values), 8):
        line = ", ".join(
            f"{float(value):.9g}f"
            for value in values[start:start + 8]
        )

        file.write(f"    {line},\n")

    file.write("};\n\n")


with open(
    HEADER_PATH,
    "w",
    encoding="utf-8",
) as file:
    file.write(
        "#ifndef LOGISTIC_REGRESSION_CLASSIFIER_MODEL_H\n"
    )
    file.write(
        "#define LOGISTIC_REGRESSION_CLASSIFIER_MODEL_H\n\n"
    )
    file.write("#include <math.h>\n\n")

    file.write(f"#define LR_WL {WINDOW_LEN}\n")
    file.write(f"#define LR_NF {N_FEATURES}\n")
    file.write(f"#define LR_INPUT {INPUT_SIZE}\n")
    file.write(f"#define LR_CLASSES {N_CLASSES}\n")
    file.write(
        f"#define LR_DIV {SCALE_DIVISOR:.9g}f\n\n"
    )

    write_float_array(
        file,
        "LR_MEAN",
        scaler_mean,
    )
    write_float_array(
        file,
        "LR_SCALE",
        scaler_scale,
    )
    write_float_array(
        file,
        "LR_WEIGHT",
        weights,
    )
    write_float_array(
        file,
        "LR_BIAS",
        bias,
    )

    c_source = r'''
static inline int logistic_regression_predict(
    const float raw[LR_WL][LR_NF],
    float probability[LR_CLASSES],
    float *confidence)
{
    float logits[LR_CLASSES];

    for (int class_index = 0;
         class_index < LR_CLASSES;
         class_index++)
    {
        float sum = LR_BIAS[class_index];

        for (int time_index = 0;
             time_index < LR_WL;
             time_index++)
        {
            for (int feature_index = 0;
                 feature_index < LR_NF;
                 feature_index++)
            {
                int flat_index =
                    time_index * LR_NF
                    + feature_index;

                float physical =
                    raw[time_index][feature_index]
                    / LR_DIV;

                float scaled =
                    LR_SCALE[flat_index] > 0.0f
                    ? (
                        physical
                        - LR_MEAN[flat_index]
                    ) / LR_SCALE[flat_index]
                    : 0.0f;

                sum += scaled
                    * LR_WEIGHT[
                        class_index * LR_INPUT
                        + flat_index
                    ];
            }
        }

        logits[class_index] = sum;
    }

    float maximum_logit = logits[0];

    for (int class_index = 1;
         class_index < LR_CLASSES;
         class_index++)
    {
        if (logits[class_index] > maximum_logit)
        {
            maximum_logit = logits[class_index];
        }
    }

    float denominator = 0.0f;

    for (int class_index = 0;
         class_index < LR_CLASSES;
         class_index++)
    {
        probability[class_index] =
            expf(
                logits[class_index]
                - maximum_logit
            );

        denominator += probability[class_index];
    }

    int predicted_class = 0;
    float maximum_probability = 0.0f;

    for (int class_index = 0;
         class_index < LR_CLASSES;
         class_index++)
    {
        probability[class_index] /= denominator;

        if (
            probability[class_index]
            > maximum_probability
        )
        {
            maximum_probability =
                probability[class_index];

            predicted_class = class_index;
        }
    }

    *confidence = maximum_probability;

    return predicted_class;
}

#endif
'''

    file.write(c_source)

print("Saved:", HEADER_PATH)
print(
    "Header size KB:",
    round(os.path.getsize(HEADER_PATH) / 1024, 2),
)


In [ ]:
# Generate Logistic Regression parameter header
from pathlib import Path
from IPython.display import FileLink, display

HEADER_PATH = (
    Path(FINAL_DIR) / "logistic_regression_params.h"
)

labels_c = ",\n".join(
    f'    "{label}"' for label in FINAL_LABELS
)

header = f"""#ifndef LOGISTIC_REGRESSION_PARAMS_H
#define LOGISTIC_REGRESSION_PARAMS_H

#define LR_WINDOW_LEN       {WINDOW_LEN}U
#define LR_WINDOW_STEP      {WINDOW_STEP}U
#define LR_N_FEATURES       {N_FEATURES}U
#define LR_INPUT_SIZE       {INPUT_SIZE}U
#define LR_N_CLASSES        {N_CLASSES}U
#define LR_ROW_PERIOD_SEC   {ROW_PERIOD_SEC:.10f}f

/*
 * StandardScaler is embedded inside the ONNX model.
 * Supply live physical FLOAT32 values directly.
 */
#define LR_EXTERNAL_SCALING  0U
#define LR_LIVE_DIVISION     0U

static const char *const LR_LABELS[LR_N_CLASSES] = {{
{labels_c}
}};

#endif /* LOGISTIC_REGRESSION_PARAMS_H */
"""

HEADER_PATH.write_text(header, encoding="utf-8")

print("Created:", HEADER_PATH)
display(FileLink(str(HEADER_PATH)))

In [ ]:
# ==========================================================
# CELL 16 — verify direct C-style calculation
# ==========================================================
def direct_python_prediction(
    flattened_window,
):
    scaled = (
        flattened_window
        - scaler_mean
    ) / scaler_scale

    logits = (
        weights @ scaled
        + bias
    )

    shifted_logits = logits - np.max(logits)

    probabilities = np.exp(shifted_logits)
    probabilities /= np.sum(probabilities)

    return int(np.argmax(probabilities))


number_check = min(100, len(X_all))

sklearn_predictions = final_model.predict(
    X_all[:number_check]
)

direct_predictions = np.asarray([
    direct_python_prediction(window)
    for window in X_all[:number_check]
])

if not np.array_equal(
    sklearn_predictions,
    direct_predictions,
):
    raise RuntimeError(
        "Direct Logistic Regression calculation "
        "does not match sklearn."
    )

print(
    "Direct C-style prediction verified on",
    number_check,
    "windows.",
)

In [ ]:
# ==========================================================
# CELL 17 — manual classification test
# ==========================================================
def manual_classification_prediction(
    test_file,
):
    model_path = os.path.abspath(
        os.path.join(
            "logistic_regression_output",
            "final_model",
            "logistic_regression_classifier.pkl",
        )
    )

    print("Loading model from:", model_path)

    if not os.path.exists(
        model_path
    ):
        raise FileNotFoundError(
            f"Model file not found:\n"
            f"{model_path}\n\n"
            "Run Cell 11 and Cell 12 first."
        )

    model = joblib.load(
        model_path
    )

    test_df = read_one(
        test_file
    )

    X_test, starts = make_windows_from_array(
        test_df.to_numpy(
            dtype=np.float32
        )
    )

    if (
        X_test is None
        or len(X_test) == 0
    ):
        raise RuntimeError(
            "The selected file does not contain enough "
            "valid rows for one 30-row window."
        )

    predictions = model.predict(
        X_test
    )

    probabilities = model.predict_proba(
        X_test
    )

    confidence = np.max(
        probabilities,
        axis=1,
    )

    result = pd.DataFrame({
        "window": np.arange(
            len(X_test)
        ),
        "start_row": starts,
        "time_sec": (
            starts + WINDOW_LEN / 2.0
        ) * ROW_PERIOD_SEC,
        "pred_class_id": predictions,
        "pred_label": [
            FINAL_LABELS[
                int(index)
            ]
            for index in predictions
        ],
        "confidence": confidence,
    })

    for class_id, label in enumerate(
        FINAL_LABELS
    ):
        result[
            f"probability_{label}"
        ] = probabilities[
            :,
            class_id,
        ]

    majority_class_id = int(
        result[
            "pred_class_id"
        ].value_counts().idxmax()
    )

    majority_label = FINAL_LABELS[
        majority_class_id
    ]

    stem = Path(
        test_file
    ).stem

    manual_test_dir = os.path.abspath(
        os.path.join(
            "logistic_regression_output",
            "manual_test_results",
        )
    )

    os.makedirs(
        manual_test_dir,
        exist_ok=True,
    )

    output_csv = os.path.join(
        manual_test_dir,
        f"manual_test_{stem}.csv",
    )

    result.to_csv(
        output_csv,
        index=False,
    )

    print("\n" + "=" * 60)
    print("MANUAL LOGISTIC REGRESSION TEST")
    print("=" * 60)
    print("File:", test_file)
    print("Total windows:", len(result))
    print(
        "Majority prediction:",
        majority_label,
    )
    print(
        "Mean confidence:",
        float(
            np.mean(
                confidence
            )
        ),
    )
    print("Saved CSV:", output_csv)

    print("\nWindow predictions:")
    print(
        result[
            "pred_label"
        ].value_counts()
    )

    # ======================================================
    # Manual-file prediction timeline
    # ======================================================
    plt.figure(
        figsize=(12, 4)
    )

    plt.step(
        result["time_sec"],
        result["pred_class_id"],
        where="mid",
    )

    plt.yticks(
        np.arange(
            N_CLASSES
        ),
        FINAL_LABELS,
    )

    plt.xlabel("Time [s]")
    plt.ylabel("Predicted class")

    plt.title(
        f"Logistic Regression — {stem}"
    )

    plt.tight_layout()

    timeline_path = os.path.join(
        manual_test_dir,
        f"manual_test_{stem}_timeline.png",
    )

    plt.savefig(
        timeline_path,
        dpi=150,
    )

    plt.show()

    print(
        "Saved timeline:",
        timeline_path,
    )

    return result

In [ ]:
# ==========================================================
# CELL 18 — select manual test file
# ==========================================================
import tkinter as tk
from tkinter import filedialog

root = tk.Tk()
root.withdraw()
root.attributes("-topmost", True)

TEST_FILE = filedialog.askopenfilename(
    title="Select motor CSV or Excel file",
    filetypes=[
        ("Excel files", "*.xlsx"),
        ("CSV files", "*.csv"),
        ("All files", "*.*"),
    ],
)

root.destroy()

if TEST_FILE:
    manual_result = manual_classification_prediction(
        TEST_FILE
    )
else:
    print("No file selected.")
